In [9]:
# Probe for step 2: one vita at a time, to try a prompt, a model or a
# glossary change by hand. The batch run is `python run_step.py 2` -- this
# notebook imports the very functions it uses, so what is tried here is exactly
# what the batch does. Why the step works the way it does is in README.md.
from run_step import STEPS, describe_reasoning, prepare

# Another model: prepare(STEPS[2], model="qwen3.8-27b"). The reasoning models
# think by default, which takes minutes per vita -- thinking=False turns that
# off and reasoning_effort=... sets how much there is, in the levels that model
# knows (helper_functions.STYLES; anything else is refused rather than ignored).
MODEL ="openai-gpt-oss-120b"

run = prepare(STEPS[2], model=MODEL)
#run = prepare(STEPS[2], model=MODEL, thinking=False)
#run = prepare(STEPS[2], model=MODEL, reasoning_effort="low")
print(f"{len(run.keys())} vitae, model {run.model} -- {describe_reasoning(run)}")

156 vitae, model openai-gpt-oss-120b -- thinking left at the model's default


In [10]:
volume, nr = 5, 885

# what the model will be shown, without calling it
ahead = run.preview(volume, nr)
print(ahead["text"])
print("-" * 100)
for occurrence in ahead["occurrences"]:
    offered = ahead["candidates"][occurrence.abbreviation]
    print(f"[[{occurrence.id}]] {occurrence.matched}: {', '.join(offered) or '(none)'}")

Brunswic Brunswicen. Halberstadensis et Hildesemensis diocc.
abbas et monasterium s. Egidii B. ordo sancti Benedicti Halberstadensis diocesis: de conserv. 30. iun. 1435 S 307 228vs.
par. ecclesia sancti Andree B. Hildesemensis diocesis Ludolpho Quirre archidiaconus in Stockem in eccl. Hildesemensis et rector d. parochialis ecclesia supplic. : de indulg. 30. iun. 1435 S 309 203r.
decanus, capitulum et singuli canonici collegiata ecclesia sancti Blasii B. unius de notabilioribus collegiata eccl. Saxonie Ottone, Wilhelmo et Hinrico Brunswicen. et Luneborgen. ducibus, patron. etiam supplic. : de incorp. parochialis ecclesia in Woden Weden Hildesemensis diocesis 4 marca argenti fabrice d. ecclesia sancti Blasii 2 marca argenti 13. october 1438 S 350 168vs.
proconsules, consules et universitas opidum B. Hildesemensis et Halberstadensis diocc.: de conserv. privilegium de non evocando eis a Sigismundo R.I. conc. et a Martino V. conf. 26. iun. 1436 S 323 235vs, exec.: abbas monasterium sancti P

In [11]:
# the whole request, system prompt first
print(run.step.prompt)
print("=" * 100)
print(ahead["prompt"])

**Role:** You are a historian specializing in medieval church history with expert knowledge of the Latin abbreviations used in the papal registers.

**Task:** You will receive a Latin text in which some abbreviations are marked as `[[id|abbreviation]]`, together with a list of expansion candidates for each id. For every id, select the candidate that best fits the grammatical and semantic context of the surrounding text.

**Instructions:**

1. **Choices:** Choose exactly one candidate per id, from the given candidates only.

2. **Base forms:** Return the chosen candidate exactly as it is written in the candidate list. Do not inflect, alter, or extend it — the grammatical form is adjusted in a later processing step.

3. **Occurrences:** The same abbreviation can require different expansions at different places in the text; judge each occurrence in its own context. If you are unsure, prefer the expansion most commonly associated with that abbreviation in medieval Latin church documents.



In [12]:
# one model call for this vita
result = run.process(volume, nr)
print(result["text"])

Brunswic Brunswicen. Halberstadensis et Hildesemensis diocc.
abbas et monasterium sanctus Egidii B. ordo sancti Benedicti Halberstadensis diocesis: de conservare 30. iunius 1435 S 307 228vs.
parochialis ecclesia sancti Andree B. Hildesemensis diocesis Ludolpho Quirre archidiaconus in Stockem in eccl. Hildesemensis et rector dominus parochialis ecclesia supplicatio : de indulgentia 30. iunius 1435 S 309 203r.
decanus, capitulum et singuli canonici collegiata ecclesia sancti Blasii B. unius de notabilioribus collegiata eccl. Saxonie Ottone, Wilhelmo et Hinrico Brunswicen. et Luneborgen. ducibus, patronatus etiam supplicatio : de incorporatio parochialis ecclesia in Woden Weden Hildesemensis diocesis 4 marca argenti fabrice dies ecclesia sancti Blasii 2 marca argenti 13. october 1438 S 350 168vs.
proconsules, consules et universitas opidum B. Hildesemensis et Halberstadensis diocc.: de conservare privilegium de non evocando eis a Sigismundo Romanorum Imperator concessio et a Martino V. co

In [13]:
for choice in result["record"]["choices"]:
    print(f"[[{choice['id']}]] {choice['abbreviation']} → {choice['choice']}")

print()
for error in result["record"]["errors"]:
    print(error["message"])

[[1]] s. → sanctus
[[2]] conserv. → conservare
[[3]] iun. → iunius
[[4]] par. → parochialis
[[5]] d. → dominus
[[6]] supplic. → supplicatio
[[7]] indulg. → indulgentia
[[8]] iun. → iunius
[[9]] patron. → patronatus
[[10]] supplic. → supplicatio
[[11]] incorp. → incorporatio
[[12]] d. → dies
[[13]] conserv. → conservare
[[14]] R.I. → Romanorum Imperator
[[15]] conc. → concessio
[[16]] conf. → confirmatio
[[17]] iun. → iunius
[[18]] exec. → executio

